In [11]:
import os
import sys
import re
import zipfile
import shutil
from collections import defaultdict

In [12]:
folder = input('Enter path to folder containing zip files: ').strip()
folder = os.path.abspath(folder)
 
output_folder = input('Enter path for merged output files: ').strip()
output_folder = os.path.abspath(output_folder)
 
# Validate folder exists
if not os.path.exists(folder):
    print(f'ERROR: Folder not found: {folder}')
    sys.exit(1)
 
# Find all zip files
zip_files = sorted([f for f in os.listdir(folder) if f.endswith('.zip')])
 
if not zip_files:
    print(f'ERROR: No zip files found in {folder}')
    sys.exit(1)
 
print(f'\nFound {len(zip_files)} zip files in {folder}:')
for z in zip_files:
    size = os.path.getsize(os.path.join(folder, z))
    print(f'  {z} ({size / (1024*1024):.2f} MB)')
 
print(f'\nMerged output will be saved to: {output_folder}')
 

Enter path to folder containing zip files:  ../../../data/IoT/11 Nov 2020/
Enter path for merged output files:  ../data/merged_raw_data



Found 39 zip files in /users/PAS1266/ralston168/data/IoT/11 Nov 2020:
  01-5.175.8717.aad2.zip (49.59 MB)
  02-31.132.3c0b.bc11.zip (32.14 MB)
  03-31.132.d063.8013.zip (0.00 MB)
  04-46.246.fe1d.9416.zip (0.00 MB)
  05-76.73.fa3a.5d9f.zip (0.00 MB)
  06-92.240.6ddd.5c03.zip (178.17 MB)
  07-213.184.2d21.9dd9.zip (105.55 MB)
  08-154.16.513e.1a2a.zip (212.53 MB)
  09-213.184.50de.192b.zip (125.46 MB)
  10-173.225.d4bd.03e4.zip (189.90 MB)
  11-209.200.ae9c.e04e.zip (181.08 MB)
  12-213.184.ecbf.7293.zip (131.72 MB)
  13-93.190.abcd.1234.zip (3.21 MB)
  14-94.229.2f7c.94dd.zip (194.57 MB)
  16-77.78.541a.814d.zip (75.91 MB)
  17-77.78.e513.a78b.zip (0.00 MB)
  18-94.229.d716.62a3.zip (175.99 MB)
  19-108.62.acb9.b7ae.zip (37.80 MB)
  20-109.200.3d7c.3d77.zip (91.16 MB)
  21-109.200.2806.636a.zip (0.00 MB)
  22-154.16.4513.6c50.zip (0.00 MB)
  23-173.225.1539.32ec.zip (65.15 MB)
  24-173.225.c128.a067.zip (0.00 MB)
  25-173.225.f157.adf6.zip (0.00 MB)
  26-173.225.49be.6699.zip (0.00 MB

In [13]:
def get_log_type(filename):
    """
    Extract log type from filename.
    e.g. '5.175.8717.aad2_00001_20170724102704-conn.logreplaced.log' -> 'conn'
    e.g. '5.175.8717.aad2_00001_20170724102704-packet_filter.logreplaced.log' -> 'packet_filter'
    """
    match = re.search(r'-([a-zA-Z_]+)\.logreplaced\.log$', filename)
    if match:
        return match.group(1)
    return None
 
 
temp_folder = os.path.join(folder, '_temp_extracted')
os.makedirs(output_folder, exist_ok=True)
 
total_zips_processed = 0
total_files_merged = 0
 
for i, zip_filename in enumerate(zip_files, 1):
    print(f'\n[{i}/{len(zip_files)}] Processing: {zip_filename}')
 
    # Create a subfolder in output named after the zip file (without .zip)
    zip_name = os.path.splitext(zip_filename)[0]
    zip_output_folder = os.path.join(output_folder, zip_name)
 
    try:
        # -- Unzip --
        os.makedirs(temp_folder, exist_ok=True)
        zip_path = os.path.join(folder, zip_filename)
 
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(temp_folder)
            num_extracted = len(z.namelist())
            print(f'  Unzipped: {num_extracted} files')
 
        # -- Group files by log type --
        log_groups = defaultdict(list)
        unrecognised = 0
 
        for filename in os.listdir(temp_folder):
            log_type = get_log_type(filename)
            if log_type:
                log_groups[log_type].append(filename)
            else:
                unrecognised += 1
 
        if not log_groups:
            print(f'  WARNING: No recognisable log files found in {zip_filename}, skipping')
            continue
 
        if unrecognised:
            print(f'  Skipping {unrecognised} unrecognised files')
 
        # -- Merge each log type into zip's own output subfolder --
        os.makedirs(zip_output_folder, exist_ok=True)
 
        for log_type, files in sorted(log_groups.items()):
            files.sort()  # Sort chronologically by timestamp in filename
            merged_filepath = os.path.join(zip_output_folder, f'all-{log_type}.log')
 
            with open(merged_filepath, 'a') as outfile:
                for filename in files:
                    filepath = os.path.join(temp_folder, filename)
                    with open(filepath, 'r', errors='replace') as infile:
                        for line in infile:
                            outfile.write(line)
 
            print(f'  Merged {len(files):4} {log_type} files → {zip_name}/all-{log_type}.log')
            total_files_merged += len(files)
 
        total_zips_processed += 1
 
    except zipfile.BadZipFile:
        print(f'  SKIPPED (bad zip): {zip_filename}')
 
    except Exception as e:
        print(f'  ERROR processing {zip_filename}: {e}')
 
    finally:
        # Always clean up temp folder even if something goes wrong
        if os.path.exists(temp_folder):
            shutil.rmtree(temp_folder)
            print(f'  Cleaned up temp files')


[1/39] Processing: 01-5.175.8717.aad2.zip


KeyboardInterrupt: 

In [ ]:
print('\n' + '=' * 60)
print(f'Zip files processed: {total_zips_processed}/{len(zip_files)}')
print(f'Total files merged:  {total_files_merged}')
print(f'\nOutput folder structure:')
for zip_name in sorted(os.listdir(output_folder)):
    zip_out = os.path.join(output_folder, zip_name)
    if os.path.isdir(zip_out):
        print(f'\n  {zip_name}/')
        for f in sorted(os.listdir(zip_out)):
            size = os.path.getsize(os.path.join(zip_out, f))
            print(f'    {f} ({size / (1024*1024):.2f} MB)')
print('\nDone!')